In [1]:
# Import essential libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt
import dgl
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.manifold import TSNE
import os # Thêm thư viện OS để tạo thư mục
from utils.attack_algo_utils import *
from utils.graph_utils import *
from utils.basic_utils import *
from utils.model_utils import *
# Đảm bảo tệp graph_utils.py nằm trong cùng thư mục

print("Đã import các thư viện.")

Đã import các thư viện.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
import networkx as nx
import dgl
import torch
import time



# --- 1. Tạo dữ liệu cho Đồ thị 1 (Mạng Công ty) ---
G_1 = nx.DiGraph()
G_1.add_nodes_from(["Data Server", "Pad", "Web Server", "Host 1", "Host 2", "Host 3", "File Server", "Router"])

node_attributes = {
    "Data Server": {"state": 0, "priority": 2}, "Pad": {"state": 0, "priority": 1},
    "Web Server": {"state": 0, "priority": 1}, "Host 1": {"state": 0, "priority": 1},
    "Host 2": {"state": 0, "priority": 0}, "Host 3": {"state": 0, "priority": 0},
    "File Server": {"state": 0, "priority": 0}, "Router": {"state": 0, "priority": 2}
}
nx.set_node_attributes(G_1, node_attributes)

# Danh sách cạnh thô
raw_edges_1 = [
    ("Pad", "Host 1", {"user": 0.6, "root": 0.48}), ("Pad", "Host 2", {"user": 0.32, "root": 0.32}),
    ("Pad", "Host 3", {"user": 0.32, "root": 0.32}), ("Pad", "Web Server", {"user": 0.8, "root": 0.6}),
    ("Host 1", "Pad", {"user": 0.6, "root": 0.6}), ("Host 1", "Web Server", {"user": 0.8, "root": 0.6}),
    ("Host 1", "Host 2", {"user": 0.32, "root": 0.32}), ("Host 1", "Host 3", {"user": 0.32, "root": 0.32}),
    ("Host 2", "Host 3", {"user": 0.8, "root": 0.8}), ("Host 2", "File Server", {"user": 0.8, "root": 0.6}),
    ("Host 2", "Data Server", {"user": 0.8, "root": 0.6}), ("Host 3", "Host 2", {"user": 0.8, "root": 0.8}),
    ("Host 3", "File Server", {"user": 0.8, "root": 0.6}), ("Host 3", "Data Server", {"user": 0.8, "root": 0.6}),
    ("Web Server", "File Server", {"user": 0.8, "root": 0.04}), ("Web Server", "Data Server", {"user": 0.8, "root": 0.04}),
    ("File Server", "Data Server", {"user": 0.8, "root": 0.04}), ("Data Server", "File Server", {"user": 0.6, "root": 0.02}),
    ("Router", "Web Server", {"user": 0.9, "root": 0.9}), ("Web Server", "Router", {"user": 0.9, "root": 0.9}),
    ("Router", "Data Server", {"user": 0.9, "root": 0.1}), ("Data Server", "Router", {"user": 0.9, "root": 0.9}),
    ("Pad", "Router", {"user": 0.7, "root": 0.5}),
]

# Xử lý timestamp và thêm vào đồ thị
sorted_edges_1 = add_timestamp_to_edges(raw_edges_1)
G_1.add_edges_from(sorted_edges_1)

# Tạo DGL Graph
g1, node_order, node_map = build_dgl(
    nx_graph=G_1,
    sorted_edges=sorted_edges_1,
    node_feat_keys=['state', 'priority'],
    edge_feat_keys=['user', 'root']
)


# --- TÁCH FEATURE (G1) ---
nfeats1 = torch.tensor([[G_1.nodes[n]['state'], G_1.nodes[n]['priority']] for n in node_order], dtype=torch.float32)
efeats1 = torch.tensor([[d['user'], d['root']] for u, v, d in G_1.edges(data=True)], dtype=torch.float32)

g1.ndata['h'] = nfeats1
g1.edata['h'] = efeats1


# --- 2. Tạo Đồ thị 2 (Mạng Văn phòng) ---
G_2 = nx.DiGraph()
G_2.add_nodes_from([
    "WAN", "DMZ Server", "User PC 1", "User PC 2",
    "WiFi AP", "Internal DB", "User PC 3", "Printer"
])
node_attr_2 = {
    "WAN": {"state": 0, "priority": 1},
    "DMZ Server": {"state": 1, "priority": 1},
    "User PC 1": {"state": 0, "priority": 0}, "User PC 2": {"state": 0, "priority": 0},
    "WiFi AP": {"state": 0, "priority": 0}, "Internal DB": {"state": 0, "priority": 1},
    "User PC 3": {"state": 0, "priority": 2}, "Printer": {"state": 0, "priority": 2}
}
nx.set_node_attributes(G_2, node_attr_2)

raw_edges_2 = [
    ("WAN", "DMZ Server", {"user": 0.9, "root": 0.8}),
    ("DMZ Server", "WiFi AP", {"user": 0.8, "root": 0.5}),
    ("DMZ Server", "Internal DB", {"user": 0.7, "root": 0.6}),
    ("WiFi AP", "User PC 1", {"user": 0.9, "root": 0.1}),
    ("WiFi AP", "User PC 2", {"user": 0.9, "root": 0.1}),
    ("WiFi AP", "User PC 3", {"user": 0.9, "root": 0.1}),
    ("User PC 1", "User PC 2", {"user": 0.3, "root": 0.6}),
    ("User PC 1", "Internal DB", {"user": 0.5, "root": 0.3}),
    ("User PC 2", "Internal DB", {"user": 0.5, "root": 0.3}),
    ("User PC 3", "Internal DB", {"user": 0.5, "root": 0.3}),
    ("User PC 1", "Printer", {"user": 0.8, "root": 0.1}),
    ("User PC 2", "Printer", {"user": 0.8, "root": 0.1}),
    ("User PC 3", "Printer", {"user": 0.8, "root": 0.1}),
]

# Xử lý timestamp cho G2
sorted_edges_2 = add_timestamp_to_edges(raw_edges_2)
G_2.add_edges_from(sorted_edges_2)

g2, node_order2, node_map2 = build_dgl(
    nx_graph=G_2,
    sorted_edges=sorted_edges_2,
    node_feat_keys=['state', 'priority'],
    edge_feat_keys=['user', 'root']
)

# --- TÁCH FEATURE (G2) ---
nfeats2 = torch.tensor([[G_2.nodes[n]['state'], G_2.nodes[n]['priority']] for n in node_order2], dtype=torch.float32)
efeats2 = torch.tensor([[d['user'], d['root']] for u, v, d in G_2.edges(data=True)], dtype=torch.float32)

g2.ndata['h'] = nfeats2
g2.edata['h'] = efeats2

# --- 3. In kết quả kiểm tra ---
print("--- ĐỒ THỊ 1 (MẠNG CÔNG TY) ---")
print(f"Đã tạo G1: {g1.num_nodes()} nodes, {g1.num_edges()} edges.")
print(f"Shape efeats1: {g1.edata['h'].shape}")

# Kiểm tra 5 cạnh đầu tiên để thấy sự chênh lệch 0.1s
print("--- KIỂM TRA TIMESTAMP (5 cạnh đầu theo thời gian) ---")

# Bước 1: Lấy tất cả cạnh ra list
all_edges_check = list(G_1.edges(data=True))

# Bước 2: SORT list này theo timestamp (Vì G_1.edges() mặc định sort theo tên Node)
all_edges_check.sort(key=lambda x: x[2]['timestamp'])

# Bước 3: Lấy mốc thời gian nhỏ nhất làm chuẩn
start_ts = all_edges_check[0][2]['timestamp']

# Bước 4: In ra 5 cạnh đầu tiên
for i in range(5):
    u, v, d = all_edges_check[i]
    ts = d['timestamp']
    diff = ts - start_ts
    # Dùng :.1f để làm tròn khớp với 0.1s
    print(f"Edge {i}: {u:12} -> {v:12} | Time: {ts:.4f} | Diff: {diff:.1f}s")

--- ĐỒ THỊ 1 (MẠNG CÔNG TY) ---
Đã tạo G1: 8 nodes, 23 edges.
Shape efeats1: torch.Size([23, 2])
--- KIỂM TRA TIMESTAMP (5 cạnh đầu theo thời gian) ---
Edge 0: Pad          -> Host 1       | Time: 1766762460.1891 | Diff: 0.0s
Edge 1: Pad          -> Host 2       | Time: 1766762460.2891 | Diff: 0.1s
Edge 2: Pad          -> Host 3       | Time: 1766762460.3891 | Diff: 0.2s
Edge 3: Pad          -> Web Server   | Time: 1766762460.4891 | Diff: 0.3s
Edge 4: Host 1       -> Pad          | Time: 1766762460.5891 | Diff: 0.4s


In [4]:

# 1. Định nghĩa các hằng số
MAX_N_FEATURES = 2
MAX_E_FEATURES = 2

def build_batch_tensor(feats, max_dim):    
    return feats[:, :max_dim]

# 4. Tạo Batch huấn luyện (Sử dụng logic mới)
print("--- Bắt đầu chuẩn hóa & đệm batch ---")

# Xử lý Node Features
nfeats_batch = build_batch_tensor(nfeats1, MAX_N_FEATURES)
nfeats_batch = nfeats_batch.to(device)

# Xử lý Edge Features
efeats_batch = build_batch_tensor(efeats1, MAX_E_FEATURES)
efeats_batch = efeats_batch.to(device)



# Batch các đồ thị
g_batch = dgl.batch([g1])
g_batch = g_batch.to(device)

print(f"\nĐã tạo batch huấn luyện: {g_batch.num_nodes()} nodes, {g_batch.num_edges()} edges")
print(f"Batch feature dims (đã đệm): Node={nfeats_batch.shape}, Edge={efeats_batch.shape}")

--- Bắt đầu chuẩn hóa & đệm batch ---

Đã tạo batch huấn luyện: 8 nodes, 23 edges
Batch feature dims (đã đệm): Node=torch.Size([8, 2]), Edge=torch.Size([23, 2])


In [5]:
# --- Training loop (SỬA ĐỔI VỚI PADDING CHUẨN HÓA) ---

# Lấy kích thước đầu vào TỐI ĐA (MAX_FEATURES)
NDIM_IN = MAX_N_FEATURES # 50
EDIM = MAX_E_FEATURES    # 50

# Lấy các tham số từ cell cũ của bạn
N_HIDDEN = 16
N_OUT = 24
N_LAYERS = 2
EPOCHS = 2000 
LEARNING_RATE = 0.001


# Khởi tạo encoder và mô hình DGI (SỬ DỤNG CÁC LỚP MỚI)
encoder = EGraphSAGE(
    NDIM_IN, EDIM, N_HIDDEN, N_OUT, N_LAYERS, 
    F.leaky_relu, device
)
dgi_model = DGI(encoder)
dgi_model = dgi_model.to(device)

# Khởi tạo optimizer
optimizer = torch.optim.Adam(dgi_model.parameters(), lr=LEARNING_RATE)

print("\n--- Bắt đầu quá trình huấn luyện DGI ---")
print(f"Kích thước đầu vào mô hình: Node={NDIM_IN}, Edge={EDIM}")

for epoch in range(EPOCHS):
    dgi_model.train()  # Chuyển mô hình sang chế độ huấn luyện
    optimizer.zero_grad()

    loss = dgi_model(g_batch, nfeats_batch, efeats_batch)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0: 
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}')

print("--- Huấn luyện hoàn tất! ---")


--- Bắt đầu quá trình huấn luyện DGI ---
Kích thước đầu vào mô hình: Node=2, Edge=2
Epoch [200/2000], Loss: 0.4508
Epoch [400/2000], Loss: 2.8106
Epoch [600/2000], Loss: 0.2560
Epoch [800/2000], Loss: 0.4849
Epoch [1000/2000], Loss: 0.0236
Epoch [1200/2000], Loss: 0.1092
Epoch [1400/2000], Loss: 0.0155
Epoch [1600/2000], Loss: 0.3650
Epoch [1800/2000], Loss: 0.0129
Epoch [2000/2000], Loss: 0.3328
--- Huấn luyện hoàn tất! ---


In [6]:
import torch
import yaml
import os

# Đảm bảo thư mục tồn tại
os.makedirs("graphs", exist_ok=True)

# ... (Giả sử các biến NDIM_IN, EDIM... đã được định nghĩa ở trên) ...

# Định nghĩa cấu hình Model
model_config = {
    "NDIM_IN": NDIM_IN,
    "EDIM": EDIM,
    "N_HIDDEN": N_HIDDEN,
    "N_OUT": N_OUT,
    "N_LAYERS": N_LAYERS,
}

# --- PHẦN THAY ĐỔI: Lưu Config bằng YAML ---
CONFIG_PATH = "graphs/model_config.yaml" # Đổi đuôi file

with open(CONFIG_PATH, 'w') as f:
    # default_flow_style=False để file yaml xuống dòng đẹp mắt
    yaml.dump(model_config, f, default_flow_style=False)

print(f"Đã lưu Cấu hình Model (YAML) vào: {CONFIG_PATH}")

# --- PHẦN GIỮ NGUYÊN: Lưu Trọng số Model (vẫn dùng torch.save) ---
MODEL_STATE_PATH = "graphs/dgi_model_state_dict.pth"
torch.save(dgi_model.state_dict(), MODEL_STATE_PATH)
print(f"Đã lưu Trọng số Model vào: {MODEL_STATE_PATH}")

Đã lưu Cấu hình Model (YAML) vào: graphs/model_config.yaml
Đã lưu Trọng số Model vào: graphs/dgi_model_state_dict.pth
